## Chains 
Chains makes your life easier.   
In an AI application you have several steps - prompts, models, outputs, conditional tasks/parallel task etc...

Chains can handle this, so we need to trigger the chain once and the whole sequence of tasks run as expected

Three types of tasks:
1. Linear Sequential Chain
2. Parallel Chain 
3. Conditional Chain

### Linear Sequential Chain 

In [6]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

In [2]:
# model
llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V4-Pro",
    task='text-generation',
    temperature=0.5
)

model = ChatHuggingFace(llm = llm)

In [3]:
#Create pydantic model
class Character(BaseModel):
    name: str = Field(..., description="Give the actual name of Character")
    age: int = Field(..., description="Give age of character")
    gender: Literal['Male','Female'] = Field(..., description="Give the gender of character")

pydantic_parser = PydanticOutputParser(pydantic_object=Character)

template = PromptTemplate(
    template="Give acutal name, age, gender of this fictional character : {character}.\n {format_instructions}",
    input_variables=['character'],
    partial_variables={'format_instructions': pydantic_parser.get_format_instructions()}
)

chain = template | model | pydantic_parser
result = chain.invoke({'character':'spider man'})

In [5]:
print(result)
chain.get_graph().print_ascii()

name='Peter Parker' age=28 gender='Male'
    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
  +-----------------+    
  | ChatHuggingFace |    
  +-----------------+    
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
      +-----------+      
      | Character |      
      +-----------+      


### Parallel chain

In [13]:
from langchain_core.runnables import RunnableParallel #runnable to run chains in parallel
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

template1 = ChatPromptTemplate(
    messages=[
        ('system', 'You are a machine learning expert. Your task is to give a 5-15 notes on given text'),
        ('user', 'Give notes on\n{text}')
])

template2 = ChatPromptTemplate(
    messages=[
        ('system','You are a quiz generator. Your task is to generate 5 questions quiz (multiple choice or fill in the blanks) from the text given'),
        ('user','Notes:\n{text}')
])

template3 = ChatPromptTemplate(
    messages=[
        ('system','Your task is to merge theory:\n{theory} \n\nand quiz:\n{quiz}')
])

parallel_chain = RunnableParallel(
    {'theory': template1 | model | parser,
    'quiz': template2 | model | parser})

sequential_chain = template3 | model | parser
merged_chain = parallel_chain | sequential_chain 

In [14]:
text = "Convolutional Neural Networks (CNNs), also known as ConvNets, are neural network architectures inspired by the human visual system and are widely used in computer vision tasks. They are designed to process structured grid-like data, especially images by capturing spatial relationships between pixels.Automatically learn hierarchical features through convolution operations, from simple edges and textures to complex shapes and objects.Detect objects at different positions within an image, ensuring robustness to spatial variations.Reduce computational complexity by processing local regions instead of the entire image at once."
result = merged_chain.invoke({
    'text': text})

In [16]:
from IPython.display import Markdown, display
display(Markdown(result))

# Convolutional Neural Networks (CNNs): Theory and Quiz

## Theory
Convolutional Neural Networks (CNNs) are a specialized class of deep learning models designed to process data with a grid-like topology, most notably images. Inspired by the organization of the animal visual cortex, CNNs automatically and adaptively learn spatial hierarchies of features through multiple layers. The architecture typically consists of three main types of layers: convolutional layers, pooling layers, and fully connected layers.

**Convolutional Layers** apply a set of learnable filters (kernels) that slide across the input image, performing element-wise multiplication and summation to produce feature maps. This operation exploits local connectivity, meaning each neuron is connected only to a small region of the input, drastically reducing the number of parameters compared to fully connected networks. As a result, CNNs can detect simple patterns like edges and textures in early layers, and progressively combine them into more complex shapes, object parts, and finally entire objects in deeper layers.

**Pooling Layers** (e.g., max pooling) downsample the feature maps, reducing spatial dimensions and computational load while providing a form of translation invariance. This makes the network robust to small shifts and distortions in the input.

**Fully Connected Layers** at the end of the network integrate the high-level features learned by the convolutional and pooling layers to perform classification or regression tasks.

Key advantages of CNNs include:
- **Parameter sharing**: The same filter is used across the entire image, making the network efficient and equivariant to translations.
- **Sparse interactions**: Only local regions are processed at a time, reducing memory and computation.
- **Automatic feature extraction**: CNNs learn relevant features directly from raw data, eliminating the need for manual feature engineering.

These properties make CNNs exceptionally effective for tasks such as image classification, object detection, semantic segmentation, and even video analysis.

---

## Quiz

1. What type of data are Convolutional Neural Networks (CNNs) especially designed to process?  
A) Sequential text data  
B) Structured grid-like data, especially images  
C) Audio waveforms  
D) Unstructured linguistic data  

2. CNNs automatically learn hierarchical features, progressing from simple edges and textures to ___________ and objects.  
(Fill in the blank)  

3. True or False: CNNs are unable to handle objects appearing at different positions in an image.  
A) True  
B) False  

4. How do CNNs reduce computational complexity?  
A) By processing the entire image at

In [19]:
print(merged_chain.get_graph().draw_ascii())

                +----------------------------+                   
                | Parallel<theory,quiz>Input |                   
                +----------------------------+                   
                    ***                  ***                     
                ****                        ****                 
              **                                **               
+--------------------+                   +--------------------+  
| ChatPromptTemplate |                   | ChatPromptTemplate |  
+--------------------+                   +--------------------+  
           *                                        *            
           *                                        *            
           *                                        *            
  +-----------------+                     +-----------------+    
  | ChatHuggingFace |                     | ChatHuggingFace |    
  +-----------------+                     +-----------------+    
          

### Conditional Chain

In [54]:
from langchain_core.runnables import RunnableBranch
from langchain_core.runnables import RunnableLambda #RunnableLambda converts lambda function to runnable so that we can use it as chain
from langchain_core.output_parsers import PydanticOutputParser

In [66]:
class Code(BaseModel):
    code: str = Field(..., description="Give simple and compressed python code for required task")
pydantic_parser1 = PydanticOutputParser(pydantic_object=code)


class Explanation(BaseModel):
    code: str = Field(..., description="Give simple and compressed python code for required task in less than 20 lines")
    explanation: str = Field(..., description="Give simple explanation of code in short")
pydantic_parser2 = PydanticOutputParser(pydantic_object=Explanation)


prompt1 = PromptTemplate(
    template="Generate a small (around 10-30 lines) python code for given task: \n {task} \n\n {format_instructions}",
    input_variables=['task'],
    partial_variables={'format_instructions':pydantic_parser1.get_format_instructions()}
    )

#IF code is more than 20 lines
prompt2 = PromptTemplate(
    template="Create two field 'Code' and 'Explanation', compress the code to 20 or less than 20 lines (if possible) and put it in 'code' field. And Explain the code in short and put it in 'explanation' field.\nCode:\n {code} \n\n {format_instructions}",
    input_variables=['code'],
    partial_variables={'format_instructions':pydantic_parser2.get_format_instructions()}
)

#If code is already less than 20 lines
prompt3 = PromptTemplate(
    template="Create two field 'Code' and 'Explanation', put code in 'code' field. And Explain the code in short and put it in 'explanation' field.\nCode:\n{code} \n\n {format_instructions}",
    input_variables=['code'],
    partial_variables={'format_instructions':pydantic_parser2.get_format_instructions()}
)


code_chain = prompt1 | model | pydantic_parser1
'''
Syntax: 
condition, chain-if-condition-is-true
condition, chain-if-condition-is-true
default chain (if no condition is statisfied)
'''
conditional_chain = RunnableBranch(
    (lambda x:len(x.code.split("\n"))>=20, prompt2 | model | pydantic_parser2),
    prompt3 | model | pydantic_parser2
)

merged_chain = code_chain | conditional_chain

In [67]:
result = merged_chain.invoke({'task':'make a simple calculator'})

In [71]:
print(result.code)
print(result.explanation)

while True:
    try:
        expr = input('Enter expression (or q to quit): ')
        if expr.lower() == 'q':
            break
        result = eval(expr)
        print('Result:', result)
    except Exception as e:
        print('Error:', e)
A simple calculator that continuously prompts for a Python expression, evaluates it with eval(), prints the result, and quits when 'q' is entered. Errors are caught and displayed.


In [72]:
merged_chain.get_graph().print_ascii()

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
  +-----------------+    
  | ChatHuggingFace |    
  +-----------------+    
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +--------------+     
